# Fine-grid job audit v3

Controlla lo stato di `graphs_fine_grid` per ogni città considerando tutte le trasformazioni previste:

- `original`
- traduzioni cardinali `trans_*`
- rotazioni `rot_*`
- scale anisotrope `scale_ns_*` e `scale_ew_*`

Una variante con `on_land=False` viene considerata **processata ma rigettata**: non va rilanciata.
Una variante va rilanciata solo se è attesa ma manca da `fine_grid_stats.csv`.


In [19]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 240)
pd.set_option("display.width", 200)

ROOT = Path("/home/fbellisardi/code/topolity")
DATA_ROOT = ROOT / "data" / "data_processed"
SCRIPT = ROOT / "pipeline_production" / "dem_extractor_fine_grid.py"

# Same configuration as your bash defaults
STEP_METERS = 500
NUM_POINTS = 5
ROT_ANGLES = [-20, -15, -10, -5, 5, 10, 15, 20]
NS_SCALES = [1.02, 1.05, 1.08, 1.12]
EW_SCALES = [1.02, 1.05, 1.08, 1.12]

# Suggested rerun settings
TIME_LIMIT = "168:00"
MEM_GB = 128
CPUS = 4
WORKERS = 1
LAND_CHECK = "sample"
LOW_MEMORY = True
USE_FUA = True
DEM_MODE = "tree"

DATA_ROOT

PosixPath('/home/fbellisardi/code/topolity/data/data_processed')

In [20]:
def normalize_rotation_angles(angles_deg):
    normalized = []
    for angle in angles_deg or []:
        a = float(angle) % 360.0
        if a > 180.0:
            a -= 360.0
        normalized.append(a)
    return sorted(set(normalized))


def format_scale_token(scale_factor):
    return f"{float(scale_factor):.3f}".replace(".", "p")


def variant_sort_key(v):
    v = str(v)
    if v == "original":
        return (0, 0, 0, "")
    m = re.match(r"trans_(\d+)m_a([+-]\d+)", v)
    if m:
        return (1, int(m.group(1)), int(m.group(2)), v)
    m = re.match(r"rot_([+-]\d+p\d+)deg", v)
    if m:
        val = float(m.group(1).replace("p", "."))
        return (2, abs(val), val, v)
    m = re.match(r"scale_(ns|ew)_(\d+p\d+)", v)
    if m:
        axis = 0 if m.group(1) == "ns" else 1
        val = float(m.group(2).replace("p", "."))
        return (3, axis, val, v)
    return (9, 0, 0, v)


def expected_rotation_variants(rotation_angles):
    variants = []
    for angle in normalize_rotation_angles(rotation_angles):
        if np.isclose(angle, 0.0):
            continue
        variants.append(f"rot_{angle:+.2f}deg".replace(".", "p"))
    return variants


def expected_scale_variants(ns_scales, ew_scales):
    variants = []
    for scale in sorted(set(float(s) for s in ns_scales)):
        if not np.isclose(scale, 1.0):
            variants.append(f"scale_ns_{format_scale_token(scale)}")
    for scale in sorted(set(float(s) for s in ew_scales)):
        if not np.isclose(scale, 1.0):
            variants.append(f"scale_ew_{format_scale_token(scale)}")
    return variants


def expected_translation_variants_from_stats_or_default(stats_df=None, step_meters=500, num_points=5):
    # The code may produce names such as 1499m due to int rounding.
    # Therefore, include both the default expected names and any translation names already present in stats.
    default = ["original"]
    for i in range(1, num_points + 1):
        d = i * step_meters
        for a in [0, 90, 180, 270]:
            default.append(f"trans_{d}m_a{a:+04d}")

    if stats_df is None or stats_df.empty or "variant" not in stats_df.columns:
        return default

    stats_vars = stats_df["variant"].dropna().astype(str)
    trans_seen = [v for v in stats_vars if v == "original" or v.startswith("trans_")]
    return sorted(set(default) | set(trans_seen), key=variant_sort_key)


def expected_variants_for_city(stats_df=None):
    translations = expected_translation_variants_from_stats_or_default(stats_df, STEP_METERS, NUM_POINTS)
    rotations = expected_rotation_variants(ROT_ANGLES)
    scales = expected_scale_variants(NS_SCALES, EW_SCALES)
    return sorted(set(translations + rotations + scales), key=variant_sort_key)


def list_city_dirs(data_root=DATA_ROOT):
    return sorted([p for p in data_root.iterdir() if p.is_dir() and (p / "data_useful.csv").exists()])


def read_stats(city_dir):
    stats_path = city_dir / "graphs_fine_grid" / "fine_grid_stats.csv"
    if not stats_path.exists() or stats_path.stat().st_size == 0:
        return pd.DataFrame(), stats_path
    try:
        return pd.read_csv(stats_path), stats_path
    except Exception as e:
        print(f"[warning] could not read {stats_path}: {e}")
        return pd.DataFrame(), stats_path


def pkl_variants(city_dir):
    gdir = city_dir / "graphs_fine_grid"
    out = set()
    if not gdir.exists():
        return out
    for p in gdir.glob("graph_*.pkl"):
        out.add(p.name[len("graph_"):-len(".pkl")])
    return out


def latest_file_info(city_dir):
    gdir = city_dir / "graphs_fine_grid"
    if not gdir.exists():
        return None, None
    files = [p for p in gdir.iterdir() if p.is_file()]
    if not files:
        return None, None
    latest = max(files, key=lambda p: p.stat().st_mtime)
    return datetime.fromtimestamp(latest.stat().st_mtime), latest.name


def bool_series_on_land(s):
    return s.astype(str).str.lower().map({
        "true": True, "false": False,
        "1": True, "0": False,
        "yes": True, "no": False,
    })


def compact_join(values, max_chars=800):
    txt = ", ".join(list(values))
    if len(txt) <= max_chars:
        return txt
    return txt[:max_chars] + "..."


In [21]:
def audit_city(city_dir):
    city = city_dir.name
    stats_df, stats_path = read_stats(city_dir)
    pkl_vars = pkl_variants(city_dir)
    latest_mtime, latest_file = latest_file_info(city_dir)

    expected = set(expected_variants_for_city(stats_df))
    stats_vars = set(stats_df["variant"].dropna().astype(str)) if not stats_df.empty and "variant" in stats_df.columns else set()

    sea_rejected = set()
    valid_stats = set()
    unknown_land_stats = set()

    if not stats_df.empty and "variant" in stats_df.columns:
        tmp = stats_df.copy()
        if "on_land" in tmp.columns:
            tmp["_on_land_bool"] = bool_series_on_land(tmp["on_land"])
            sea_rejected = set(tmp.loc[tmp["_on_land_bool"] == False, "variant"].dropna().astype(str))
            valid_stats = set(tmp.loc[tmp["_on_land_bool"] == True, "variant"].dropna().astype(str))
            unknown_land_stats = set(tmp.loc[tmp["_on_land_bool"].isna(), "variant"].dropna().astype(str))
        else:
            valid_stats = stats_vars

    # Processed = present in stats, including on_land=False rejected variants
    processed = stats_vars
    missing_real = expected - processed

    valid_without_pkl = valid_stats - pkl_vars
    pkl_without_stats = pkl_vars - stats_vars

    extra_stats = stats_vars - expected
    extra_pkl = pkl_vars - expected

    if len(stats_vars) == 0 and len(pkl_vars) == 0:
        status = "not_started"
    elif len(missing_real) == 0 and len(valid_without_pkl) == 0:
        status = "done_or_rejected"
    elif len(missing_real) == 0 and len(valid_without_pkl) > 0:
        status = "stats_without_pkl_check"
    else:
        status = "partial_to_resume"

    return {
        "city": city,
        "status": status,
        "expected_total": len(expected),
        "expected_translations": len([v for v in expected if v == "original" or v.startswith("trans_")]),
        "expected_rotations": len([v for v in expected if v.startswith("rot_")]),
        "expected_scales": len([v for v in expected if v.startswith("scale_")]),
        "stats_rows": len(stats_df),
        "stats_variants": len(stats_vars),
        "pkl_count": len(pkl_vars),
        "valid_completed_count": len(valid_stats & pkl_vars),
        "sea_rejected_count": len(sea_rejected),
        "missing_real_count": len(missing_real),
        "valid_without_pkl_count": len(valid_without_pkl),
        "pkl_without_stats_count": len(pkl_without_stats),
        "extra_stats_count": len(extra_stats),
        "extra_pkl_count": len(extra_pkl),
        "latest_mtime": latest_mtime,
        "latest_file": latest_file,
        "missing_real_variants": compact_join(sorted(missing_real, key=variant_sort_key)),
        "valid_without_pkl_variants": compact_join(sorted(valid_without_pkl, key=variant_sort_key)),
        "sea_rejected_variants": compact_join(sorted(sea_rejected, key=variant_sort_key)),
        "extra_stats_variants": compact_join(sorted(extra_stats, key=variant_sort_key), max_chars=350),
        "extra_pkl_variants": compact_join(sorted(extra_pkl, key=variant_sort_key), max_chars=350),
    }


rows = [audit_city(city_dir) for city_dir in list_city_dirs(DATA_ROOT)]
audit = pd.DataFrame(rows).sort_values(
    ["status", "missing_real_count", "valid_without_pkl_count", "city"],
    ascending=[True, False, False, True],
).reset_index(drop=True)

audit


,city,status,expected_total,expected_translations,expected_rotations,expected_scales,stats_rows,stats_variants,pkl_count,valid_completed_count,sea_rejected_count,missing_real_count,valid_without_pkl_count,pkl_without_stats_count,extra_stats_count,extra_pkl_count,latest_mtime,latest_file,missing_real_variants,valid_without_pkl_variants,sea_rejected_variants,extra_stats_variants,extra_pkl_variants
0,amsterdam,done_or_rejected,37,21,8,8,37,37,37,37,0,0,0,0,0,0,2026-04-29 11:26:15.192732,fine_grid_gravitational_work.pdf,,,,,
1,bandung,done_or_rejected,37,21,8,8,37,37,37,37,0,0,0,0,0,0,2026-04-29 12:06:15.273475,fine_grid_gravitational_work.pdf,,,,,
2,bogota,done_or_rejected,37,21,8,8,37,37,37,37,0,0,0,0,0,0,2026-04-29 12:19:30.474786,fine_grid_gravitational_work.pdf,,,,,
3,caracas,done_or_rejected,37,21,8,8,37,37,36,36,1,0,0,0,0,0,2026-04-29 12:35:42.857298,fine_grid_gravitational_work.pdf,,,rot_-20p00deg,,
4,chicago,done_or_rejected,37,21,8,8,37,37,37,37,0,0,0,0,0,0,2026-05-03 09:42:27.658141,fine_grid_stats.csv,,,,,
5,moscou,done_or_rejected,37,21,8,8,49,49,49,49,0,0,0,0,12,12,2026-05-01 05:48:46.407484,fine_grid_stats.csv,,,,"rot_-0p50deg, rot_+0p50deg, rot_-1p00deg, rot_+1p00deg, rot_-2p00deg, rot_+2p00deg, rot_-25p00deg, rot_+25p00deg, rot_-30p00deg, rot_+30p00deg, rot_-45p00deg, rot_+45p00deg","rot_-0p50deg, rot_+0p50deg, rot_-1p00deg, rot_+1p00deg, rot_-2p00deg, rot_+2p00deg, rot_-25p00deg, rot_+25p00deg, rot_-30p00deg, rot_+30p00deg, rot_-45p00deg, rot_+45p00deg"
6,santiago,done_or_rejected,37,21,8,8,49,46,159,46,0,0,0,113,9,122,2026-04-30 13:08:52.055308,fine_grid_gravitational_work.pdf,,,,"rot_+0p50deg, rot_+1p00deg, rot_+2p00deg, rot_-25p00deg, rot_+25p00deg, rot_-30p00deg, rot_+30p00deg, rot_-45p00deg, rot_+45p00deg","trans_50m_a+000, trans_50m_a+090, trans_50m_a+180, trans_50m_a+270, trans_100m_a+000, trans_100m_a+090, trans_100m_a+180, trans_100m_a+270, trans_150m_a+000, trans_150m_a+090, trans_150m_a+180, trans_150m_a+270, trans_200m_a+000, trans_..."
7,hongkong,partial_to_resume,37,21,8,8,1,1,0,0,1,36,0,0,0,0,2026-04-18 22:15:49.567302,fine_grid_stats.csv,"trans_500m_a+000, trans_500m_a+090, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_...",,original,,
8,vancouver,partial_to_resume,39,23,8,8,3,3,0,0,3,36,0,0,0,0,2026-04-20 23:34:00.510090,fine_grid_stats.csv,"trans_500m_a+000, trans_500m_a+090, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_...",,"original, trans_499m_a+000, trans_999m_a+000",,
9,pekin,partial_to_resume,37,21,8,8,2,2,2,2,0,35,0,0,0,0,2026-04-30 05:43:18.941112,fine_grid_gravitational_work.pdf,"trans_500m_a+090, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m...",,,,


## Città da rilanciare

Queste sono le città con varianti attese ma non presenti in `fine_grid_stats.csv`.
Il conteggio include anche rotazioni e scale transformations.


In [22]:
to_rerun = audit[audit["missing_real_count"] > 0].copy()
to_rerun = to_rerun.sort_values(["missing_real_count", "city"], ascending=[False, True]).reset_index(drop=True)

cols = [
    "city", "status",
    "expected_total", "expected_translations", "expected_rotations", "expected_scales",
    "missing_real_count", "valid_completed_count", "sea_rejected_count",
    "stats_rows", "pkl_count", "latest_mtime", "latest_file",
    "missing_real_variants",
]
to_rerun[cols]


,city,status,expected_total,expected_translations,expected_rotations,expected_scales,missing_real_count,valid_completed_count,sea_rejected_count,stats_rows,pkl_count,latest_mtime,latest_file,missing_real_variants
0,hongkong,partial_to_resume,37,21,8,8,36,0,1,1,0,2026-04-18 22:15:49.567302,fine_grid_stats.csv,"trans_500m_a+000, trans_500m_a+090, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_..."
1,vancouver,partial_to_resume,39,23,8,8,36,0,3,3,0,2026-04-20 23:34:00.510090,fine_grid_stats.csv,"trans_500m_a+000, trans_500m_a+090, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_..."
2,pekin,partial_to_resume,37,21,8,8,35,2,0,2,2,2026-04-30 05:43:18.941112,fine_grid_gravitational_work.pdf,"trans_500m_a+090, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m..."
3,riodejaneiro,partial_to_resume,37,21,8,8,35,0,2,2,0,2026-04-20 03:23:16.377474,fine_grid_stats.csv,"trans_500m_a+000, trans_500m_a+090, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m..."
4,sandiego,partial_to_resume,37,21,8,8,34,0,3,3,0,2026-04-20 15:51:53.013618,fine_grid_stats.csv,"trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m_a+090, trans_2000..."
5,sanfrancisco,partial_to_resume,37,21,8,8,34,0,3,3,0,2026-04-20 14:14:03.660885,fine_grid_stats.csv,"trans_500m_a+000, trans_500m_a+090, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m_a+090, trans_2000..."
6,seoul,partial_to_resume,37,21,8,8,34,0,3,3,0,2026-04-20 16:15:35.519376,fine_grid_stats.csv,"trans_500m_a+000, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m..."
7,dublin,partial_to_resume,40,24,8,8,33,0,7,7,0,2026-04-19 00:15:56.029286,fine_grid_stats.csv,"trans_500m_a+000, trans_500m_a+090, trans_500m_a+180, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+270, trans_2000m_a+000, trans_2000m_a+090, trans_2000m..."
8,istanbul,partial_to_resume,38,22,8,8,31,0,7,7,0,2026-04-19 02:49:24.606321,fine_grid_stats.csv,"trans_500m_a+000, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m_a+180, trans_2000m_a+270, trans_2500m..."
9,lisbon,partial_to_resume,37,21,8,8,31,0,6,6,0,2026-04-19 09:03:55.692942,fine_grid_stats.csv,"trans_500m_a+000, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+270, trans_2000m_a+000, trans_2000m_a+090, trans_2000m_a+180, trans_2000m_a+270, trans_2500..."


## Città complete o rigettate correttamente

Qui `missing_real_count = 0`. Se `sea_rejected_count` è alto è normale per città costiere/isole.


In [23]:
done = audit[audit["missing_real_count"] == 0].copy()
done = done.sort_values(["status", "sea_rejected_count", "city"], ascending=[True, False, True]).reset_index(drop=True)

done[[
    "city", "status", "expected_total", "valid_completed_count",
    "sea_rejected_count", "valid_without_pkl_count",
    "stats_rows", "pkl_count", "latest_mtime", "latest_file"
]]


,city,status,expected_total,valid_completed_count,sea_rejected_count,valid_without_pkl_count,stats_rows,pkl_count,latest_mtime,latest_file
0,caracas,done_or_rejected,37,36,1,0,37,36,2026-04-29 12:35:42.857298,fine_grid_gravitational_work.pdf
1,amsterdam,done_or_rejected,37,37,0,0,37,37,2026-04-29 11:26:15.192732,fine_grid_gravitational_work.pdf
2,bandung,done_or_rejected,37,37,0,0,37,37,2026-04-29 12:06:15.273475,fine_grid_gravitational_work.pdf
3,bogota,done_or_rejected,37,37,0,0,37,37,2026-04-29 12:19:30.474786,fine_grid_gravitational_work.pdf
4,chicago,done_or_rejected,37,37,0,0,37,37,2026-05-03 09:42:27.658141,fine_grid_stats.csv
5,moscou,done_or_rejected,37,49,0,0,49,49,2026-05-01 05:48:46.407484,fine_grid_stats.csv
6,santiago,done_or_rejected,37,46,0,0,49,159,2026-04-30 13:08:52.055308,fine_grid_gravitational_work.pdf


## Casi sospetti

`valid_without_pkl_count > 0` significa: nel CSV la variante è `on_land=True`, ma manca il pickle. Queste vanno controllate o rilanciate.


In [24]:
suspicious = audit[
    (audit["valid_without_pkl_count"] > 0) |
    (audit["pkl_without_stats_count"] > 0)
].copy()

suspicious[[
    "city", "status", "valid_without_pkl_count", "pkl_without_stats_count",
    "valid_without_pkl_variants", "latest_mtime", "latest_file"
]]


,city,status,valid_without_pkl_count,pkl_without_stats_count,valid_without_pkl_variants,latest_mtime,latest_file
6,santiago,done_or_rejected,0,113,,2026-04-30 13:08:52.055308,fine_grid_gravitational_work.pdf
20,barcelone,partial_to_resume,0,3,,2026-04-29 11:33:23.995022,fine_grid_gravitational_work.pdf
27,rome,partial_to_resume,0,33,,2026-04-30 08:33:28.831811,fine_grid_gravitational_work.pdf
37,paris,partial_to_resume,0,66,,2026-05-06 15:03:23.861425,fine_grid_stats.csv
53,madrid,partial_to_resume,0,72,,2026-05-06 05:19:48.600191,fine_grid_stats.csv
54,atlanta,partial_to_resume,0,136,,2026-05-03 04:24:57.963103,fine_grid_gravitational_work.pdf
56,milan,partial_to_resume,0,10,,2026-04-29 20:41:51.245169,fine_grid_gravitational_work.pdf


## Comandi di rilancio

I comandi usano `--resume`, quindi saltano ciò che è già presente nel CSV.


In [26]:
def build_rerun_command(city):
    low_memory_flag = " --low-memory" if LOW_MEMORY else ""
    fua_flag = " --use-fua" if USE_FUA else ""
    rot = " ".join(str(x) for x in ROT_ANGLES)
    ns = " ".join(str(x) for x in NS_SCALES)
    ew = " ".join(str(x) for x in EW_SCALES)

    return (
        f"runlog -t {TIME_LIMIT} -m {MEM_GB} -c {CPUS} -j fg_{city}_resume "
        f"conda run --no-capture-output -n geo_flow "
        f"python -u {SCRIPT} "
        f"--city {city} "
        f"--step-meters {STEP_METERS} "
        f"--num-points {NUM_POINTS} "
        f"--rotation-angles {rot} "
        f"--ns-scale-factors {ns} "
        f"--ew-scale-factors {ew} "
        f"--workers {WORKERS} "
        f"--seed 42 "
        f"--resume "
        f"--dem-mode {DEM_MODE}"
        f"{low_memory_flag}"
        f"{fua_flag} "
        f"--land-check {LAND_CHECK}"
    )


rerun_commands = to_rerun[["city", "missing_real_count", "missing_real_variants"]].copy()
rerun_commands["command"] = rerun_commands["city"].apply(build_rerun_command)
rerun_commands


,city,missing_real_count,missing_real_variants,command
0,hongkong,36,"trans_500m_a+000, trans_500m_a+090, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_...",runlog -t 168:00 -m 128 -c 4 -j fg_hongkong_resume conda run --no-capture-output -n geo_flow python -u /home/fbellisardi/code/topolity/pipeline_production/dem_extractor_fine_grid.py --city hongkong --step-meters 500 --num-points 5 --rot...
1,vancouver,36,"trans_500m_a+000, trans_500m_a+090, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_...",runlog -t 168:00 -m 128 -c 4 -j fg_vancouver_resume conda run --no-capture-output -n geo_flow python -u /home/fbellisardi/code/topolity/pipeline_production/dem_extractor_fine_grid.py --city vancouver --step-meters 500 --num-points 5 --r...
2,pekin,35,"trans_500m_a+090, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m...",runlog -t 168:00 -m 128 -c 4 -j fg_pekin_resume conda run --no-capture-output -n geo_flow python -u /home/fbellisardi/code/topolity/pipeline_production/dem_extractor_fine_grid.py --city pekin --step-meters 500 --num-points 5 --rotation-...
3,riodejaneiro,35,"trans_500m_a+000, trans_500m_a+090, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m...",runlog -t 168:00 -m 128 -c 4 -j fg_riodejaneiro_resume conda run --no-capture-output -n geo_flow python -u /home/fbellisardi/code/topolity/pipeline_production/dem_extractor_fine_grid.py --city riodejaneiro --step-meters 500 --num-points...
4,sandiego,34,"trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m_a+090, trans_2000...",runlog -t 168:00 -m 128 -c 4 -j fg_sandiego_resume conda run --no-capture-output -n geo_flow python -u /home/fbellisardi/code/topolity/pipeline_production/dem_extractor_fine_grid.py --city sandiego --step-meters 500 --num-points 5 --rot...
5,sanfrancisco,34,"trans_500m_a+000, trans_500m_a+090, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m_a+090, trans_2000...",runlog -t 168:00 -m 128 -c 4 -j fg_sanfrancisco_resume conda run --no-capture-output -n geo_flow python -u /home/fbellisardi/code/topolity/pipeline_production/dem_extractor_fine_grid.py --city sanfrancisco --step-meters 500 --num-points...
6,seoul,34,"trans_500m_a+000, trans_500m_a+180, trans_500m_a+270, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+180, trans_1500m_a+270, trans_2000m_a+000, trans_2000m...",runlog -t 168:00 -m 128 -c 4 -j fg_seoul_resume conda run --no-capture-output -n geo_flow python -u /home/fbellisardi/code/topolity/pipeline_production/dem_extractor_fine_grid.py --city seoul --step-meters 500 --num-points 5 --rotation-...
7,dublin,33,"trans_500m_a+000, trans_500m_a+090, trans_500m_a+180, trans_1000m_a+000, trans_1000m_a+090, trans_1000m_a+180, trans_1000m_a+270, trans_1500m_a+000, trans_1500m_a+090, trans_1500m_a+270, trans_2000m_a+000, trans_2000m_a+090, trans_2000m...",runlog -t 168:00 -m 128 -c 4 -j fg_dublin_resume conda run --no-capture-output -n geo_flow python -u /home/fbellisardi/code/topolity/pipeline_production/dem_extractor_fine_grid.py --city dublin --step-meters 500 --nu

In [27]:
OUT_DIR = ROOT / "reports" / "fine_grid_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

audit_path = OUT_DIR / "fine_grid_audit_v3_all_cities.csv"
rerun_path = OUT_DIR / "fine_grid_audit_v3_to_rerun.csv"
commands_path = OUT_DIR / "fine_grid_audit_v3_rerun_commands.sh"

audit.to_csv(audit_path, index=False)
to_rerun.to_csv(rerun_path, index=False)

with open(commands_path, "w") as f:
    f.write("#!/usr/bin/env bash\n")
    f.write("set -euo pipefail\n\n")
    for cmd in rerun_commands["command"]:
        f.write(cmd + "\n")

print("Saved:")
print(audit_path)
print(rerun_path)
print(commands_path)


Saved:
/home/fbellisardi/code/topolity/reports/fine_grid_audit/fine_grid_audit_v3_all_cities.csv
/home/fbellisardi/code/topolity/reports/fine_grid_audit/fine_grid_audit_v3_to_rerun.csv
/home/fbellisardi/code/topolity/reports/fine_grid_audit/fine_grid_audit_v3_rerun_commands.sh


## Ispezione dettagliata di una città

In [25]:
CITY = "dallas"

city_dir = DATA_ROOT / CITY
stats_df, _ = read_stats(city_dir)
expected = set(expected_variants_for_city(stats_df))
stats_vars = set(stats_df["variant"].dropna().astype(str)) if not stats_df.empty and "variant" in stats_df.columns else set()
pkl_vars = pkl_variants(city_dir)

detail = pd.DataFrame({"variant": sorted(expected | stats_vars | pkl_vars, key=variant_sort_key)})
detail["expected"] = detail["variant"].isin(expected)
detail["in_stats"] = detail["variant"].isin(stats_vars)
detail["has_pkl"] = detail["variant"].isin(pkl_vars)

if not stats_df.empty and "variant" in stats_df.columns:
    tmp = stats_df.copy()
    if "on_land" in tmp.columns:
        tmp["_on_land_bool"] = bool_series_on_land(tmp["on_land"])
        land_map = dict(zip(tmp["variant"].astype(str), tmp["_on_land_bool"]))
        detail["on_land"] = detail["variant"].map(land_map)
    else:
        detail["on_land"] = np.nan

detail["needs_rerun"] = detail["expected"] & (~detail["in_stats"])
detail.sort_values(["needs_rerun", "variant"], ascending=[False, True])


,variant,expected,in_stats,has_pkl,on_land,needs_rerun
25,rot_+10p00deg,True,False,False,NaN,True
27,rot_+15p00deg,True,False,False,NaN,True
29,rot_+20p00deg,True,False,False,NaN,True
23,rot_+5p00deg,True,False,False,NaN,True
24,rot_-10p00deg,True,False,False,NaN,True
26,rot_-15p00deg,True,False,False,NaN,True
28,rot_-20p00deg,True,False,False,NaN,True
22,rot_-5p00deg,True,False,False,NaN,True
34,scale_ew_1p020,True,False,False,NaN,True
35,scale_ew_1p050,True,False,False,NaN,True
